# PACE MiniGrid full run

Enable **GPU** and **Internet** in Kaggle. Pick **one** scoring rule in the configuration and use **Save & Run All**. The notebook fetches a pinned public source revision and installs the pinned Kaggle runtime.

- **L2** reproduces the original PACE score.
- **KL** compares the old policy to the provisional policy with forward KL divergence.
- **TVD** uses the squared Wasserstein transport distance with uniform unit costs between distinct actions.

Each full run ends at 30,000 PPO batches. OOD evaluation happens on the live policy and saves metrics after each evaluation. The reported gap is the same optimality gap used by the earlier baselines. It is measured as 1 minus mean OOD solve rate. Run outputs are packaged as a ZIP in the Kaggle working directory.

Set **SMOKE_TEST** to true for a quick trial. Runs restart from the selected seed if interrupted.


In [ ]:
from pathlib import Path

SCORE_METRIC = "l2"  # l2, kl, or tvd
SEED = 0
PROTOCOL = "algorithm1"  # algorithm1 or reuse
PROVISIONAL_ALPHA = 0.01
SMOKE_TEST = False

OOD_EVERY_ENV_STEPS = 4_096_000
OOD_CURVE_EPISODES = 20
OOD_FINAL_EPISODES = 100
OOD_LAYOUTS_JSON = ""  # Optional paper-layout JSON with the five additional grids.

REPO_URL = "https://github.com/SimforianusJonathan/PACE-Research.git"
REPO_COMMIT = "451e4cf02bb1861558cba8741919585c9ec47203"
WORK = Path("/kaggle/working")

if SCORE_METRIC not in ("l2", "kl", "tvd"):
    raise ValueError("SCORE_METRIC must be l2, kl, or tvd")
if PROTOCOL not in ("algorithm1", "reuse"):
    raise ValueError("PROTOCOL must be algorithm1 or reuse")
if OOD_LAYOUTS_JSON and not Path(OOD_LAYOUTS_JSON).is_file():
    raise FileNotFoundError(OOD_LAYOUTS_JSON)


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from textwrap import dedent
from time import perf_counter

setup_start = perf_counter()
run_root = WORK / "pace"
run_root.mkdir(parents=True, exist_ok=True)
mode = "smoke" if SMOKE_TEST else "full"
RUN = Path(tempfile.mkdtemp(prefix=f"pace-{PROTOCOL}-{SCORE_METRIC}-seed{SEED}-{mode}-", dir=run_root))
REPO = Path(tempfile.mkdtemp(prefix="pace-policy-source-"))
RUNTIME = Path("/tmp/pace-kaggle")
print(f"Run directory: {RUN}", flush=True)

smi = shutil.which("nvidia-smi")
if smi:
    subprocess.run([smi, "-L"], check=False)

env = {**os.environ, "PACE_RUNTIME_DIR": str(RUNTIME), "JAX_PLATFORMS": "cuda",
       "CUDA_VISIBLE_DEVICES": os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0],
       "XLA_PYTHON_CLIENT_PREALLOCATE": "false", "MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"}
paths = ["/usr/local/nvidia/lib64", *os.environ.get("LD_LIBRARY_PATH", "").split(os.pathsep)]
driver_paths = dict.fromkeys(path for path in paths if Path(path).is_absolute()
                            and Path(path).name not in {"compat", "stubs"}
                            and (Path(path) / "libcuda.so.1").is_file())
if driver_paths:
    env["LD_LIBRARY_PATH"] = os.pathsep.join(driver_paths)
else:
    env.pop("LD_LIBRARY_PATH", None)

def run_logged(command, name, *, append=False):
    path = RUN / name
    with path.open("a" if append else "w", encoding="utf-8") as log, subprocess.Popen(
        command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
    ) as process:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"{name} failed with exit code {code}. See {path}")

driver_check = dedent("""
    import ctypes
    cuda = ctypes.CDLL("libcuda.so.1")
    if cuda.cuInit(0):
        raise RuntimeError("CUDA driver initialization failed")
    count = ctypes.c_int()
    if cuda.cuDeviceGetCount(ctypes.byref(count)) or count.value < 1:
        raise RuntimeError("No visible CUDA device")
""")
try:
    run_logged([sys.executable, "-c", driver_check], "cuda-driver.log")
except RuntimeError:
    raise RuntimeError("Select a Kaggle GPU accelerator before running this notebook.") from None

run_logged(["git", "init", "--quiet"], "source.log")
run_logged(["git", "fetch", "--depth=1", REPO_URL, REPO_COMMIT], "source.log", append=True)
run_logged(["git", "checkout", "--detach", "FETCH_HEAD"], "source.log", append=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if actual != REPO_COMMIT:
    raise RuntimeError(f"Unexpected source revision: {actual}")
run_logged(["bash", "scripts/setup_kaggle.sh"], "setup.log")
PACE_PYTHON = str(RUNTIME / "venv/bin/python")
probe = dedent("""
    import importlib.metadata as metadata
    import json
    import jax
    import jax.numpy as jnp
    from train.pace_minigrid import PACE_SCORE_METRICS
    assert PACE_SCORE_METRICS == ("l2", "kl", "tvd")
    devices = jax.local_devices(backend="gpu")
    assert len(devices) == 1, devices
    value = jax.jit(lambda x: x @ x)(jnp.ones((16, 16))).block_until_ready()
    assert value.device.platform == "gpu" and float(value[0, 0]) == 16.0
    pin = "0f8f1284677375b889e4f13a32c9617cd009f8c4"
    origin = json.loads(metadata.distribution("jaxued").read_text("direct_url.json"))
    assert origin["vcs_info"]["commit_id"] == pin
    print({"device": str(devices[0]), "jax": jax.__version__, "jaxued_commit": pin})
""")
run_logged([PACE_PYTHON, "-c", probe], "gpu-probe.log")
setup_seconds = perf_counter() - setup_start


In [ ]:
evaluation = (
    ["--ood-every-env-steps", "25", "--curve-episodes", "2", "--final-episodes", "3",
     "--horizon", "8", "--cv-checkpoints", "1", "--cv-repeats", "2"]
    if SMOKE_TEST else
    ["--ood-every-env-steps", str(OOD_EVERY_ENV_STEPS), "--curve-episodes", str(OOD_CURVE_EPISODES),
     "--final-episodes", str(OOD_FINAL_EPISODES)]
)
command = [
    PACE_PYTHON, "-m", "train.pace_minigrid", "smoke" if SMOKE_TEST else "train",
    "--device", "gpu", "--method", "pace", "--protocol", PROTOCOL, "--score-metric", SCORE_METRIC,
    "--seed", str(SEED), "--budget", "ppo", "--final-only", "--chunk-size", "50", "--output-dir", str(RUN),
    *evaluation,
]
if not SMOKE_TEST:
    command += ["--provisional-alpha", str(PROVISIONAL_ALPHA)]
if OOD_LAYOUTS_JSON:
    command += ["--suite", "paper", "--levels-json", OOD_LAYOUTS_JSON]
run_logged(command, "train.log")

summary_path = RUN / "run_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
expected_ppo = 4 if SMOKE_TEST else 30_000
if summary["status"] != "training_complete" or summary["counts"]["ppo_batches"] != expected_ppo:
    raise RuntimeError(f"Training did not reach {expected_ppo} PPO batches. See {summary_path}")
if summary["config"]["score_metric"] != SCORE_METRIC or list(RUN.rglob("*.msgpack")):
    raise RuntimeError("Unexpected scoring rule or checkpoint artifact")
summary["source_commit"] = REPO_COMMIT
summary["timing"]["setup_seconds"] = round(setup_seconds, 2)
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")


In [ ]:
from IPython.display import FileLink, Image, display

run_logged([PACE_PYTHON, "-m", "train.pace_analysis", "aggregate",
            "--reports", str(RUN / "final_ood.json"), "--output-dir", str(RUN)], "aggregate.log")
run_logged([PACE_PYTHON, "-m", "train.pace_plot",
            "--run-dirs", str(RUN), "--output-dir", str(RUN)], "plots.log")
archive = Path(shutil.make_archive(str(WORK / RUN.name), "zip", root_dir=RUN))
metrics = json.loads((RUN / "ood_aggregate.json").read_text(encoding="utf-8"))
print({"ppo_batches": expected_ppo, "training_env_steps": summary["counts"]["env_steps"],
       "iqm": metrics["iqm"]["estimate"], "optimality_gap": metrics["optimality_gap"]["estimate"]})
display(Image(filename=str(RUN / "training_learning_curve.png")))
display(Image(filename=str(RUN / "ood_learning_curves.png")))
display(FileLink(os.path.relpath(archive, Path.cwd())))
